# Modified VIEScore for RefEdit-Bench

Scores **existing generated RefEdit outputs**; this notebook does not run the editing agent. It reads both `results/results.jsonl` and `results/images/*_output.png`, so saved images remain discoverable if a subset run replaced the JSONL summary. Scoring uses RefEdit's `metadata.edit_instruction_single`, the dataset GT polygon mask, and identical zero-padding tight source/output crops. Scores are cached per output file in `results/metrics/modified_viescore.jsonl`.

Install GEdit-Bench VIEScore dependencies in this notebook's Python environment (including `regex` and `requests`), clone [Step1X-Edit](https://github.com/stepfun-ai/Step1X-Edit), and set `VIESCORE_ROOT` to its `GEdit-Bench` directory. Set `VIESCORE_KEY_PATH` to a local file containing only your OpenAI API key on the first line. The scorer requests the `gpt-4o` API model; API usage may incur charges. Never put the key in a cell or commit its file.

In [1]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents) if (path / 'pyproject.toml').is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Open this notebook from within the MMEdit repository.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / '.env')

ANNOTATION_FILE = Path(
    os.environ.get(
        'MMEDIT_REFEDIT_ANNOTATIONS',
        '~/Datasets/MMEdit/refedit-bench/annotations.jsonl',
    )
).expanduser().resolve()
OUTPUT_DIR = Path(
    os.environ.get(
        'MMEDIT_REFEDIT_OUTPUT_DIR',
        str(PROJECT_ROOT / 'outputs/evaluation/refedit_bench'),
    )
).expanduser().resolve()
VIESCORE_ROOT = Path(os.environ.get('VIESCORE_ROOT', '')).expanduser()
VIESCORE_KEY_PATH = Path(os.environ.get('VIESCORE_KEY_PATH', '')).expanduser()

if not ANNOTATION_FILE.is_file():
    raise FileNotFoundError(f'RefEdit annotations not found: {ANNOTATION_FILE}')
if not OUTPUT_DIR.is_dir():
    raise FileNotFoundError(f'RefEdit output directory not found: {OUTPUT_DIR}')
print('Annotations:', ANNOTATION_FILE)
print('Output directory:', OUTPUT_DIR)
print('VIEScore checkout:', VIESCORE_ROOT if str(VIESCORE_ROOT) != '.' else 'not configured')
print('API key file:', VIESCORE_KEY_PATH if str(VIESCORE_KEY_PATH) != '.' else 'not configured')

Annotations: /Users/nghialt3670/Datasets/MMEdit/refedit-bench/annotations.jsonl
Output directory: /Users/nghialt3670/Projects/MMEdit/mmedit-framework/outputs/evaluation/refedit_bench
VIEScore checkout: /Users/nghialt3670/Projects/Step1X-Edit/GEdit-Bench
API key file: /Users/nghialt3670/Projects/MMEdit/mmedit-framework/.viescore.key


In [2]:
from evaluation.common import HFImageLoader, read_jsonl
from evaluation.refedit_viescore import collect_saved_outputs

records = list(read_jsonl(ANNOTATION_FILE))
if not records:
    raise ValueError(f'No RefEdit annotations found in {ANNOTATION_FILE}')
image_loader = HFImageLoader()
saved_outputs = collect_saved_outputs(
    records,
    OUTPUT_DIR / 'results' / 'results.jsonl',
    OUTPUT_DIR / 'results' / 'images',
)
print(f'Found {len(saved_outputs)} saved output images:')
for saved in saved_outputs:
    print(f"  [{saved['annotation_index']}] {saved['sample_id']} ({saved['discovered_from']})")

Found 200 saved output images:
  [0] easy:000 (results.jsonl)
  [1] easy:001 (results.jsonl)
  [2] easy:002 (results.jsonl)
  [3] easy:003 (results.jsonl)
  [4] easy:004 (results.jsonl)
  [5] easy:005 (results.jsonl)
  [6] easy:006 (results.jsonl)
  [7] easy:007 (results.jsonl)
  [8] easy:008 (results.jsonl)
  [9] easy:009 (results.jsonl)
  [10] easy:010 (results.jsonl)
  [11] easy:011 (results.jsonl)
  [12] easy:012 (results.jsonl)
  [13] easy:013 (results.jsonl)
  [14] easy:014 (results.jsonl)
  [15] easy:015 (results.jsonl)
  [16] easy:016 (results.jsonl)
  [17] easy:017 (results.jsonl)
  [18] easy:018 (results.jsonl)
  [19] easy:019 (results.jsonl)
  [20] easy:020 (results.jsonl)
  [21] easy:021 (results.jsonl)
  [22] easy:022 (results.jsonl)
  [23] easy:023 (results.jsonl)
  [24] easy:024 (results.jsonl)
  [25] easy:025 (results.jsonl)
  [26] easy:026 (results.jsonl)
  [27] easy:027 (results.jsonl)
  [28] easy:028 (results.jsonl)
  [29] easy:029 (results.jsonl)
  [30] easy:030 (re

/Users/nghialt3670/Projects/MMEdit/mmedit-framework/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Score saved outputs

Run this cell after reviewing the discovered sample list above. Unchanged images reuse cached VIEScore rows; changed or newly discovered images are evaluated and appended to the output JSONL.

## Configure VIEScore

This uses the upstream VIEScore `tie` task with the GPT-4o API model. The API key is read from the local key file; it is never displayed.

In [3]:
if not (VIESCORE_ROOT / 'viescore' / '__init__.py').is_file():
    raise FileNotFoundError(f'Set VIESCORE_ROOT to the Step1X-Edit GEdit-Bench directory; expected viescore/__init__.py under {VIESCORE_ROOT}')
if not VIESCORE_KEY_PATH.is_file():
    raise FileNotFoundError(f'Set VIESCORE_KEY_PATH to your local OpenAI API key file: {VIESCORE_KEY_PATH}')
for import_path in (VIESCORE_ROOT, VIESCORE_ROOT / 'viescore'):
    import_path_text = str(import_path.resolve())
    if import_path_text not in sys.path:
        sys.path.insert(0, import_path_text)
try:
    from viescore import VIEScore
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError('Install the VIEScore GEdit-Bench dependencies, including regex and requests, in this notebook environment.') from exc

scorer = VIEScore(backbone='gpt4o', task='tie', key_path=str(VIESCORE_KEY_PATH))
if not hasattr(scorer.model, 'model_name'):
    raise RuntimeError('This VIEScore version does not expose its API model name; cannot guarantee GPT-4o evaluation.')
scorer.model.model_name = 'gpt-4o'
print('VIEScore API model:', scorer.model.model_name)

VIEScore API model: gpt-4o


## Score saved outputs

The loop below checks annotation/dataset consistency, reuses unchanged cached rows, scores uncached outputs, and atomically rewrites the JSONL after each sample so progress is preserved.

In [4]:
import json
from tqdm.auto import tqdm
from PIL import Image
from evaluation.refedit_metrics import crop_pair_to_mask_roi, summarize_viescore_scores

metric_path = OUTPUT_DIR / 'results' / 'metrics' / 'modified_viescore.jsonl'
cached_scores = {}
if metric_path.is_file():
    for cached in read_jsonl(metric_path):
        cache_key = (cached.get('sample_id'), cached.get('output_path'), cached.get('output_size_bytes'), cached.get('output_mtime_ns'), cached.get('instruction'))
        cached_scores[cache_key] = cached

metric_rows = []
for saved in tqdm(saved_outputs, desc='Modified VIEScore', unit='sample', total=len(saved_outputs)):
    annotation_index = saved['annotation_index']
    record = records[annotation_index]
    metadata = record.get('metadata') or {}
    instruction = metadata.get('edit_instruction_single')
    if not isinstance(instruction, str) or not instruction.strip():
        raise ValueError(f"Missing metadata.edit_instruction_single for {record.get('sample_id')}")
    category = str(metadata.get('category', '')).lower()
    if category not in {'easy', 'hard'}:
        raise ValueError(f"Unexpected RefEdit category for {record.get('sample_id')}: {category!r}")
    output_path = saved['output_path']
    output_stat = output_path.stat()
    cache_key = (record.get('sample_id'), str(output_path), output_stat.st_size, output_stat.st_mtime_ns, instruction)
    if cache_key in cached_scores:
        score_row = cached_scores[cache_key]
        print(f"Using cached VIEScore for {record['sample_id']}")
    else:
        source_reference = image_loader.parse_reference(record.get('source_ref'))
        if source_reference is None:
            raise ValueError(f"Missing source_ref for {record.get('sample_id')}")
        dataset_id, hf_split, row_index, image_field = source_reference
        hf_row = image_loader.get_row(dataset_id, hf_split, row_index)
        if str(hf_row.get('category', '')).lower() != category:
            raise ValueError(f"Dataset/annotation category mismatch for {record.get('sample_id')}")
        if hf_row.get('edit_instruction_single') != instruction:
            raise ValueError(f"Dataset/annotation instruction mismatch for {record.get('sample_id')}")
        source_image = image_loader.as_image(hf_row[image_field])
        with Image.open(output_path) as generated_image:
            source_roi, edited_roi, roi_box = crop_pair_to_mask_roi(source_image, generated_image, hf_row['mask'])
        scores = scorer.evaluate([source_roi, edited_roi], instruction, extract_all_score=True)
        if not isinstance(scores, (list, tuple)) or len(scores) != 3:
            raise ValueError(f'Expected VIEScore [SC, PQ, O], got {scores!r}')
        score_row = {
            'sample_id': record['sample_id'],
            'annotation_index': annotation_index,
            'category': category,
            'instruction': instruction,
            'output_path': str(output_path),
            'output_size_bytes': output_stat.st_size,
            'output_mtime_ns': output_stat.st_mtime_ns,
            'roi_bbox_xyxy': list(roi_box),
            'sc': float(scores[0]),
            'pq': float(scores[1]),
            'o': float(scores[2]),
        }
    summarize_viescore_scores([score_row])
    metric_rows.append(score_row)
    metric_path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = metric_path.with_suffix('.jsonl.tmp')
    temporary_path.write_text(''.join(json.dumps(row, ensure_ascii=False) + '\n' for row in metric_rows), encoding='utf-8')
    temporary_path.replace(metric_path)

score_summary = summarize_viescore_scores(metric_rows)
print(f'Per-sample Modified VIEScore: {metric_path}')
display([{'category': category, **metrics} for category, metrics in score_summary.items()])

Modified VIEScore:   0%|          | 0/200 [00:00<?, ?sample/s]

Using cached VIEScore for easy:000
Using cached VIEScore for easy:001
Using cached VIEScore for easy:002
Using cached VIEScore for easy:003
Using cached VIEScore for easy:004
Using cached VIEScore for easy:005
Using cached VIEScore for easy:006
Using cached VIEScore for easy:007
Using cached VIEScore for easy:008
Using cached VIEScore for easy:009
Using cached VIEScore for easy:010
Using cached VIEScore for easy:011
Using cached VIEScore for easy:012


Modified VIEScore: 100%|██████████| 200/200 [11:56<00:00,  3.58s/sample]

Per-sample Modified VIEScore: /Users/nghialt3670/Projects/MMEdit/mmedit-framework/outputs/evaluation/refedit_bench/results/metrics/modified_viescore.jsonl


[{'category': 'easy',
  'sc': 7.09,
  'pq': 4.19,
  'o': 4.892508700632339,
  'count': 100},
 {'category': 'hard',
  'sc': 7.33,
  'pq': 3.96,
  'o': 4.974894547885823,
  'count': 100}]